# Lecture 3 — Practical: *Five Representations, One Model*
### Practical Machine Learning for Transcriptomics in Cancer Research

This practical is a **controlled experiment in representation**. You hold the model fixed — the
regularised Logistic Regression from Lecture 2 — and feed it **five different representations** of the
*same* METABRIC patients, then compare them and diagnose a deliberately leaky workflow.

The five representations:
1. **all genes** (top-2,000 by variance — the Lecture 2 baseline),
2. **variance-filtered** genes (unsupervised),
3. **differentially-expressed** genes (supervised — done *inside the fold*),
4. **biological signatures** (proliferation, ER, immune, stromal),
5. **pathway-level features** (~10 hallmark-like pathway activities).

> **The thesis you are testing:** *better biological representation often improves predictive
> performance more than changing the algorithm.* Only the representation changes; the model never does.
> The deliverable is a methods-style comparison table plus a reasoned recommendation.

#### Continuity & reminders
- **Label:** binary **recurrence** (relapse within the horizon) — *not* pCR. Carried from Lectures 1–2.
- **Leakage discipline (extended):** every *data-dependent* step (variance/DE filtering, data-driven
  selection) is fit on training data only and refit **inside each CV fold**. **Fixed, pre-published
  gene lists** (the signatures and pathways here) are largely leakage-*exempt* — they were not chosen
  using your outcome.
- **Same data cache as Lectures 1–2** — METABRIC is reused, not re-downloaded.

> **Network note.** Reuses the L1/L2 real-data loaders (cBioPortal + GEO). If the prepared cohort is in
> the shared cache it is used directly; otherwise it is regenerated (needs internet). Downloads are
> git-ignored.

## Section 0 — Setup & framing  *(≈10 min)*

In [ ]:
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import GEOparse  # noqa
except ImportError:
    _pip("GEOparse")

import os, tarfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

def _resolve_data_dir():
    """Find the shared lesson-01 datasets/ cache regardless of where Jupyter launched,
    so this Lecture-3 notebook reuses the data Lectures 1-2 already downloaded."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        lessons = parent / "lessons"
        if lessons.is_dir():
            for cand in sorted(lessons.glob("lesson01_*/practical/task/datasets")):
                cand.mkdir(parents=True, exist_ok=True); return cand
            for cand in sorted(lessons.glob("lesson01_*/practical/task")):
                d = cand / "datasets"; d.mkdir(parents=True, exist_ok=True); return d
    for c in [here.parent / "datasets", here / "datasets"]:
        if c.parent.exists():
            c.mkdir(parents=True, exist_ok=True); return c
    fb = here / "datasets"; fb.mkdir(parents=True, exist_ok=True); return fb

DATA_DIR = str(_resolve_data_dir())
print("Setup complete. Shared data cache:")
print("  ", os.path.abspath(DATA_DIR))

### Loading the prepared cohort + fixed biological inputs (shared infrastructure)

The cell below reuses the Lecture 1 loaders to rebuild the prepared cohort (HR+/HER2−, binary
recurrence label, patient-level stratified split), and defines the **fixed, pre-published** gene lists
(proliferation, ER, immune, stromal) and a small **hallmark-like gene-set collection**. These external
inputs were *not* chosen using our outcome — which is exactly why building features from them is
largely leakage-exempt. Read it, but you don't need to edit it.


In [ ]:
CBIO_URL = "https://cbioportal-datahub.s3.amazonaws.com/brca_metabric.tar.gz"

def load_metabric(data_dir=DATA_DIR):
    tar_path = os.path.join(data_dir, "brca_metabric.tar.gz")
    if not os.path.exists(tar_path):
        print("Downloading METABRIC from cBioPortal (~50 MB, one time)...")
        r = requests.get(CBIO_URL, stream=True, timeout=120); r.raise_for_status()
        with open(tar_path, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
    with tarfile.open(tar_path, "r:gz") as tf:
        expr = pd.read_csv(tf.extractfile("brca_metabric/data_mrna_illumina_microarray.txt"),
                           sep="\t", low_memory=False)
        pat = pd.read_csv(tf.extractfile("brca_metabric/data_clinical_patient.txt"),
                          sep="\t", comment="#", low_memory=False)
        smp = pd.read_csv(tf.extractfile("brca_metabric/data_clinical_sample.txt"),
                          sep="\t", comment="#", low_memory=False)
    expr = expr.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in expr.columns])
    expr = expr.dropna(subset=["Hugo_Symbol"]).set_index("Hugo_Symbol")
    expr = expr[~expr.index.duplicated(keep="first")]
    clin = pat.merge(smp, on="PATIENT_ID", how="inner", suffixes=("", "_smp"))
    if "SAMPLE_ID" in clin.columns:
        clin = clin.set_index("SAMPLE_ID")
    return expr, clin

def prepare_cohort():
    """Reproduce the Lectures 1-2 prepared cohort: HR+/HER2-, binary recurrence label,
    samples x genes matrix (top-2000-variance genes) aligned to the label."""
    expr, clin = load_metabric()
    X = expr.T.copy(); X.index.name = "SAMPLE_ID"
    common = sorted(set(X.index) & set(clin.index))
    X, clin = X.loc[common], clin.loc[common]
    hrpos = clin.get("ER_STATUS").eq("Positive") | clin.get("PR_STATUS").eq("Positive")
    her2neg = clin.get("HER2_STATUS").eq("Negative")
    mask = (hrpos & her2neg).fillna(False)
    X, clin = X.loc[mask], clin.loc[mask]
    HORIZON = 60
    status_col = next((c for c in ["RFS_STATUS", "DFS_STATUS"] if c in clin.columns), None)
    months_col = next((c for c in ["RFS_MONTHS", "DFS_MONTHS"] if c in clin.columns), None)
    recurred = clin[status_col].astype(str).str.startswith("1")
    months = pd.to_numeric(clin[months_col], errors="coerce")
    y = pd.Series(index=clin.index, dtype="float")
    y[(recurred) & (months <= HORIZON)] = 1
    y[(~recurred) & (months >= HORIZON)] = 0
    y[(recurred) & (months > HORIZON)] = 0
    keep = y.notna()
    X, clin, y = X.loc[keep], clin.loc[keep], y[keep].astype(int)
    X = X.apply(pd.to_numeric, errors="coerce")
    # keep the full ~2000-variance gene space used in L2 (label-free filter, pre-split)
    top_var = X.var(axis=0).sort_values(ascending=False).head(2000).index
    X = X[top_var]
    return X, clin, y

# --- Fixed, PRE-PUBLISHED biological inputs (not chosen using our outcome) ---
# Curated signature gene lists (illustrative, widely-used markers).
SIGNATURES = {
    "proliferation": ["MKI67", "AURKA", "CCNB1", "CCNB2", "BUB1", "TOP2A", "CDK1", "CCNE2",
                      "MYBL2", "UBE2C", "BIRC5", "RRM2", "TYMS", "CENPF", "PLK1"],
    "er_signalling": ["ESR1", "FOXA1", "GATA3", "XBP1", "BCL2", "PGR", "TFF1", "GREB1",
                      "AR", "NAT1", "MLPH"],
    "immune":        ["CD8A", "CD8B", "GZMB", "PRF1", "IFNG", "CXCL9", "CXCL10", "CD3D",
                      "CD3E", "GZMA", "NKG7", "STAT1"],
    "stromal":       ["FAP", "COL1A1", "COL1A2", "COL3A1", "ACTA2", "PDGFRB", "FN1",
                      "VIM", "THY1", "SPARC", "TIMP1"],
}
# Small hallmark-like gene-set collection (illustrative; fixed in advance).
HALLMARK_SETS = {
    "HALLMARK_E2F_TARGETS":        ["MKI67", "BUB1", "CCNB2", "AURKA", "TOP2A", "RRM2", "MYBL2", "CDK1"],
    "HALLMARK_G2M_CHECKPOINT":     ["CCNB1", "CCNB2", "PLK1", "BUB1", "CENPF", "UBE2C", "BIRC5", "CDK1"],
    "HALLMARK_ESTROGEN_EARLY":     ["ESR1", "FOXA1", "GATA3", "TFF1", "GREB1", "PGR", "XBP1"],
    "HALLMARK_ESTROGEN_LATE":      ["BCL2", "NAT1", "MLPH", "AR", "ESR1", "TFF1"],
    "HALLMARK_INTERFERON_GAMMA":   ["CXCL9", "CXCL10", "STAT1", "IFNG", "GZMB", "PRF1"],
    "HALLMARK_INFLAMMATORY":       ["CD8A", "CD3D", "CD3E", "GZMA", "NKG7", "CD8B"],
    "HALLMARK_EMT":                ["COL1A1", "COL1A2", "COL3A1", "FN1", "VIM", "SPARC", "ACTA2"],
    "HALLMARK_ANGIOGENESIS":       ["PDGFRB", "TIMP1", "FAP", "THY1", "SPARC"],
    "HALLMARK_APOPTOSIS":          ["BCL2", "BIRC5", "TIMP1", "GZMB"],
    "HALLMARK_MYC_TARGETS":        ["RRM2", "TYMS", "CCNE2", "UBE2C", "CDK1", "MYBL2"],
}

X_all, clin_all, y_all = prepare_cohort()
idx = y_all.index.to_numpy()
tr, tmp = train_test_split(idx, test_size=0.40, random_state=RANDOM_STATE, stratify=y_all.loc[idx])
va, te  = train_test_split(tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_all.loc[tmp])
X_train, y_train = X_all.loc[tr], y_all.loc[tr]
X_val,   y_val   = X_all.loc[va], y_all.loc[va]
X_test,  y_test  = X_all.loc[te], y_all.loc[te]

print(f"prepared cohort: {X_all.shape[0]} patients x {X_all.shape[1]} genes")
print(f"split: train {len(tr)} | val {len(va)} | test {len(te)}")
print(f"recurrence prevalence: {y_all.mean():.1%}")
print(f"signatures available: {list(SIGNATURES)}")
print(f"hallmark-like sets: {len(HALLMARK_SETS)}")

### The fixed model and the `evaluate()` helper (shared infrastructure)

One model for the whole notebook — the Lecture 2 regularised Logistic Regression in a pipeline — and
one `evaluate()` helper that takes a feature matrix and returns honest cross-validated ROC-AUC and
PR-AUC plus the feature count. **Only the features change between parts; the model never does.**


In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_model():
    """The fixed Lecture 2 baseline: impute -> scale -> regularised Logistic Regression."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(C=0.05, max_iter=5000, random_state=RANDOM_STATE)),
    ])

def evaluate(X_feat, y=y_train, label="(unnamed)", cv=CV):
    """Honest CV of the FIXED model on a given representation. Returns a result dict.
    The whole pipeline refits inside each fold; X_feat must already be a fold-safe representation
    (i.e. built without using the outcome, OR built inside the pipeline)."""
    roc = cross_val_score(make_model(), X_feat, y, cv=cv, scoring="roc_auc")
    pr  = cross_val_score(make_model(), X_feat, y, cv=cv, scoring="average_precision")
    return {"representation": label, "roc_auc": roc.mean(), "roc_sd": roc.std(),
            "pr_auc": pr.mean(), "n_features": X_feat.shape[1]}

RESULTS = []   # we accumulate one row per representation here
print("Fixed model and evaluate() ready. Model:", make_model().named_steps["clf"])

---
## Part 1 — Baseline: all genes  *(≈20 min)*

The Lecture 2 baseline, as our reference row: the fixed model on the all-genes matrix — capped at the
top 2,000 most-variable genes for tractability (the Lecture 2 baseline space).

> **Exercise 1.1 — the all-genes row.**
>
> Run the all-genes training matrix through `evaluate(...)` and append the result to `RESULTS`. Note
> the feature count (= all genes, i.e. the top-2,000-variance baseline) and the honest CV ROC-AUC / PR-AUC.
>
> *Hint:* `evaluate(X_train, label="all genes")`.

In [ ]:
# TODO 1.1
# Evaluate the fixed model on the full gene matrix; append to RESULTS; print ROC-AUC, PR-AUC, #features.
# res = evaluate(X_train, label="all genes"); RESULTS.append(res)


> **Discussion (Part 1).** Before going further — what do you predict happens to performance as we reduce to fewer, more meaningful features?

---
## Part 2 — Variance-filtered genes  *(≈20 min)*

An **unsupervised** filter: keep the most variable genes. It never looks at the label, so it is
low-leakage and may even be done before the split (though keeping it tidy is fine).


> **Exercise 2.1 — variance filter at two thresholds.**
>
> Keep the top-k most variable genes (try k = 500 and k = 200), evaluate each, and append rows. Use
> *training* variance to choose genes (a label-free statistic).
>
> **Exercise 2.2 (reasoning).** In a comment, state why variance filtering is low-leakage and whether
> it could be done before the split.
>
> *Hint:* `X_train.var().sort_values(ascending=False).head(k).index`.


In [ ]:
# TODO 2.1 / 2.2
# For k in (500, 200): keep the top-k most variable genes (training variance), evaluate, append.
# Then in a comment explain why variance filtering is low-leakage and whether it can precede the split.
# for k in (500, 200):
#     top = X_train.var().sort_values(ascending=False).head(k).index
#     ...


> **Discussion (Part 2).** Did dropping ~75–90% of genes hurt performance? What does that tell you about how much of the gene space was carrying signal?

---
## Part 3 — Differentially-expressed genes  *(≈30 min)*

A **supervised** filter: keep genes associated with recurrence. Because it uses the label, it **must be
done inside each CV fold** — so we put `SelectKBest` *inside the pipeline*, where `cross_val_score`
refits it per fold. We also peek at what the quick-and-leaky version would have reported.


> **Exercise 3.1 — DE selection done honestly (inside the fold).**
>
> Build a pipeline `impute → SelectKBest(f_classif, k=100) → scale → clf` and cross-validate it. Because
> selection lives *inside* the pipeline, it refits within each fold — no leakage. Append the row.
>
> **Exercise 3.2 (leakage contrast — a preview of Part 7).** Also compute the score if DE selection were
> done **once on all training data** before CV. Note the gap; we measure it properly in Part 7.
>
> *Hint:* a `Pipeline` with a `SelectKBest` step, scored with `cross_val_score`, is fold-safe.


In [ ]:
# TODO 3.1 / 3.2
# 3.1 HONEST: Pipeline(impute -> SelectKBest(f_classif, k=100) -> scale -> clf); cross_val_score it; append.
# 3.2 LEAKY preview: SelectKBest on ALL training data once, then CV the fixed reduced set; print the gap.
# K = 100
# honest_de = Pipeline([("impute",...),("select",SelectKBest(f_classif,k=K)),("scale",...),("clf",...)])


> **Discussion (Part 3).** DE filtering is in every transcriptomics paper. Why is *where* you do it (inside vs outside the fold) more important than *whether* you do it?

---
## Part 4 — Biological signature features  *(≈35 min)*

Build four interpretable scores from the **fixed, pre-published** gene lists: proliferation, ER
signalling, immune, stromal. Because the lists are external (not chosen from our outcome), building
these scores is leakage-exempt. Each score = mean of the (standardised) member genes present.


> **Exercise 4.1 — construct the four signature scores.**
>
> For each signature, z-score its member genes (those present in `X_train`) and average them into one
> column. Assemble a patients × 4 signature matrix. Inspect the sign of each score's association with
> recurrence (proliferation should be ↑ with recurrence, ER ↓).
>
> **Exercise 4.2 — evaluate the signatures-only model** and append the row (4 features).
>
> *Hint:* standardise per gene with `(g - g.mean()) / g.std()`, then `.mean(axis=1)` across members.


In [ ]:
# TODO 4.1 / 4.2
# Write signature_scores(X): for each signature, z-score present member genes and average -> one column.
# Build S_train (patients x 4); print each score's correlation with recurrence (proliferation +, ER -).
# Then evaluate(S_train, label="signatures (4)") and append.
# def signature_scores(X, signatures=SIGNATURES): ...


> **Discussion (Part 4).** A four-feature model vs the all-genes (top-2,000) model — if they perform similarly, which do you prefer, and why?

---
## Part 5 — Pathway-level features  *(≈35 min)*

Transform genes → ~10 hallmark-like **pathway activities** using a provided conceptual scorer (a simple,
documented stand-in for ssGSEA/GSVA: the mean of standardised member genes per set). Result: patients ×
~10 pathways. Same fixed model.


> **Exercise 5.1 — build the pathway matrix** (patients × ~10) and confirm the dimensional collapse.
>
> **Exercise 5.2 — evaluate the pathway model** and append the row.
>
> **Exercise 5.3 (stability).** Across bootstrap resamples, record how stable the *top* pathway (by |coef|)
> is, vs the top single gene from Part 1. Aggregated features should be more stable.
>
> *Hint:* the scorer is the same idea as `signature_scores`, applied to `HALLMARK_SETS`.


In [ ]:
# TODO 5.1 / 5.2 / 5.3
# Write pathway_scores(X) like signature_scores but over HALLMARK_SETS -> patients x ~10 matrix.
# Confirm the collapse (genes -> pathways). evaluate(P_train, ...) and append.
# 5.3: across 20 bootstraps, count the most-frequent top PATHWAY (|coef|) vs top GENE; compare stability.
# def pathway_scores(X, gene_sets=HALLMARK_SETS): ...


> **Discussion (Part 5).** Why might pathway features transfer better to the GSE6532 cohort (a different platform) than raw genes? *(Optional: try projecting onto GSE6532.)*

---
## Part 6 — Compare representations  *(≈25 min)*

Assemble the headline comparison: all five representations on the same model and metrics. This table is
the centrepiece artefact.


> **Exercise 6.1 — the comparison table & figure.**
>
> Turn `RESULTS` into a DataFrame sorted by ROC-AUC; render it, and plot ROC-AUC (with its SD) and
> PR-AUC across representations.
>
> **Exercise 6.2 (judgement).** In a comment, rank the representations on performance *and* on feature
> count / interpretability, and note where those rankings agree and disagree.


In [ ]:
# TODO 6.1 / 6.2
# Build a DataFrame from RESULTS sorted by roc_auc; print it. Plot ROC-AUC (with SD) and PR-AUC per rep.
# Then in a comment rank the representations on performance AND on #features/interpretability; note
# where the rankings agree/disagree.
# tbl = pd.DataFrame(RESULTS).sort_values("roc_auc", ascending=False)


> **Discussion (Part 6).** Did fewer, biological features match or beat all genes? Does the result support the lecture thesis *on this dataset*?

---
## Part 7 — Leakage exercise  *(the critical section, ≈25 min)*

A deliberately flawed workflow is provided. You will **diagnose** it, **measure** the optimism it injects,
and run the sobering **permuted-label** demo.


**The flawed workflow (provided):**
```
# select the 100 most recurrence-associated genes using ALL the data
sel = SelectKBest(f_classif, k=100).fit(X_all_imputed, y_all)
X_reduced = sel.transform(X_all_imputed)
# THEN cross-validate on the fixed reduced set
cv_auc = cross_val_score(model, X_reduced, y_all, cv=5, scoring="roc_auc").mean()
```


> **Exercise 7.1 (diagnose).** In a comment, identify *exactly* where this leaks and why the score is
> optimistic.
>
> **Exercise 7.2 (measure).** Run both the flawed (select-on-all-data) and correct (select-inside-fold)
> pipelines on the training data; report the AUC gap — the optimism.
>
> **Exercise 7.3 (the sobering demo).** Permute the labels (destroying all real signal) and run the
> *flawed* pipeline. Observe that it still reports a non-trivial AUC, and explain.


In [ ]:
# TODO 7.1 / 7.2 / 7.3
# 7.1 In a comment, identify exactly where the provided workflow leaks and why it is optimistic.
# 7.2 Measure: flawed (SelectKBest on all training data, then CV) vs honest (SelectKBest inside the
#     pipeline/fold). Print both CV ROC-AUCs and the optimism gap.
# 7.3 Permute y_train (destroy signal); run the FLAWED pipeline and the HONEST pipeline on permuted
#     labels; show the flawed one scores well above 0.5 while the honest one is ~0.5. Explain.
# K = 100; Xtr_imp = ... ; sel_all = SelectKBest(...).fit(Xtr_imp, y_train); ...


> **Discussion (Part 7).** If a leaked CV looks high *and* stable, how would a reviewer ever catch it from the paper? What would you demand to see?

---
## Part 8 — Biological interpretation & recommendation  *(≈20 min — the assessable deliverable)*

Write a **representation recommendation** (200–300 words): which representation you would trust for a
recurrence biomarker and why, weighing ROC-AUC/PR-AUC against feature count, stability, interpretability,
and cross-platform robustness — and what you would validate next.


**Representation recommendation (200–300 words):**

*(TODO — write your recommendation here. Use your Part 6 table and Part 5 stability result. Cover: which
representation you'd trust and why; the performance-vs-interpretability trade-off; cross-platform
robustness; what Part 7 taught you about trusting reported numbers; and what you'd validate next —
including whether your best representation adds value over a cheap clinical variable like tumour grade.)*


---
### Deliverables checklist
- [ ] Verified split + reusable `evaluate` helper (Section 0)
- [ ] All-genes baseline row (Part 1)
- [ ] Variance-filtered rows + leakage-risk reasoning (Part 2)
- [ ] DE rows computed inside the fold + honest-vs-leaky note (Part 3)
- [ ] Four signature scores + signatures-only row (Part 4)
- [ ] Pathway matrix + pathway row + stability comparison (Part 5)
- [ ] Five-representation comparison table + figure (Part 6)
- [ ] Leakage diagnosis + measured optimism gap + permuted-label demo (Part 7)
- [ ] Final representation recommendation (Part 8)

> **The message, in one line:** *better biological representation often improves predictive performance
> more than changing algorithms.* You now have the controlled experiment to back it — and the leakage
> diagnostic to defend it.
